# Language Model Fundamentals

Notebook demonstrating token IDs, shifted inputs/targets, vocabulary-sized logits, softmax probabilities, and cross-entropy loss.

In [ ]:
import torch
import torch.nn.functional as F # functions for nn operations

For this 5 tiny token vocabulary, the model needs to output 5 logits at each token position.

In [ ]:
# Tiny pseudo token vocabulary
vocab = {
    "I": 0,
    "like": 1,
    "machine": 2,
    "learning": 3,
    ".": 4
}

print(vocab)
print("Vocabulary size:", len(vocab))

We encode the token sequence into tensors so the nn can perform (matrix) operations with it.

In [ ]:
# Token sequence
tokens = ["I", "like", "machine", "learning", "."]

# Encode sequence
token_ids = [vocab[tokens] for tokens in tokens] # for each token in sequence, convert to ID

print("Token sequence:", tokens)
print("Token IDs:", token_ids)

# Convert token IDs -> tensor
sequence = torch.tensor(token_ids) # 1D tensor

print("\nSequence tensor:", sequence)
print("Sequence shape:", sequence.shape)

This shifted sequence represents how the training lines up. In training, the will use context available at each position to predict the next token, not just the last.

In [ ]:
# Shift sequence to get input/target
x = sequence[:-1] # input, ignore last token
y = sequence[1:] # target, ignore first (0th) token

print("Input: ", x)
print("Target:", y)

The logit tensor has shape [sequence length, vocab size] or conventionally [T, V]. In our case, the model needs to output a tensor with shape [4, 5].

In [ ]:
# Fake model outputs -> logits
logits = torch.tensor([
    [0.2, 2.5, 0.4, -1.0, 0.1],
    [0.1, 0.3, 2.2, 0.5, -0.4],
    [-0.2, 0.4, 0.6, 2.8, 0.1],
    [0.0, -0.3, 0.2, 0.4, 2.4]
])

print("Logits:")
print(logits)

print("Shape:", logits.shape)

Softmax turns these raw logit scores into a probability distribution. PyTorch does not have a built-in temperature parameter in softmax. Temperature is scaled by diving the aforementioned logits by the temperature value (T) before applying softmax.

https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.softmax.html

In [ ]:
# Convert logits into probabilities
probabilities = F.softmax(
    logits,
    dim=-1 # applies softmax across last dimension (vocab size)
)

print("Probabilities:")
print(probabilities)

print("\nRow sums:")
print(probabilities.sum(dim=-1))

In [ ]:
# Highest probability token for each token position
predicted_ids = probabilities.argmax(dim=-1)

print("Predicted token IDs from (fake) logits:", predicted_ids)
print(y)

Below is a manually written softmax function that takes in `n` logits, computes the sum of every exponentiated logit, and dividing each logit by that sum to compute the probability distribution.

In [ ]:
def manual_softmax(logits, temperature=1.0):
    scaled_logits = logits / temperature # scale logits by temperature

    exp_logits = torch.exp(scaled_logits) # exponentiate logits

    probabilities = exp_logits / exp_logits.sum() # divide every logit by sum of logits

    return probabilities

# Calculate softmax for first row
row = logits[0]

manual_probs = manual_softmax(row)

pytorch_probs = F.softmax(row, dim=-1)

# Verify manual and PyTorch softmax results
print("Manual:")
print(manual_probs)

print("\nPyTorch:")
print(pytorch_probs)

Playing around with the temperature allows us to directly see the distribution get shaper for large temperature values and evening out for small values.

In [ ]:
default_temp = manual_softmax(row) # default temperature=1.0
small_temp = manual_softmax(row, temperature=0.1)
large_temp = manual_softmax(row, temperature=10.0)

print("Default temperature:")
print(default_temp)

print("\nSmall temperature:")
print(small_temp)

print("\nLarge temperature:")
print(large_temp)

A more numerical stable implementation of the manual softmax function could be that for every vocab-sized output in each token position, we subtract the highest logit from every logit. 

The largest logit becomes 0 and everything else becomes negative, this keeps every exponentiated logit bounded above at e^0 = 1. This avoids overflow from the previous function, as logits such as 1000 -> e^1000 could become too big for python/pytorch to handle.

The reason why softmax still computes the same scores even after subtracting every logit is because it only cares about relative differences between every logit. For example:

$ \frac{e^1}{e^0} = e^1 = \frac{e^0}{e^{-1}} $ keeps the same ratio

Softmax does not care about the absolute “height" of the logits. It cares about how far apart they are from each other.

In [ ]:
def improved_softmax(logits, temperature=1.0):
    scaled_logits = logits / temperature # scale logits by temperature

    scaled_logits -= scaled_logits.max() # subtract max of logits from every logit

    exp_logits = torch.exp(scaled_logits) # exponentiate logits

    probabilities = exp_logits / exp_logits.sum() # divide every logit by sum of logits

    return probabilities

# Calculate softmax for first row
row = logits[0]

manual_probs = manual_softmax(row)

pytorch_probs = F.softmax(row, dim=-1)

# Verify manual and PyTorch softmax results
print("Manual:")
print(manual_probs)

print("\nPyTorch:")
print(pytorch_probs)

We'll also briefly write manual code for cross-entropy before using PyTorch's cross_entropy() function, which is just the negative log of the correct token's probability (for that token position).

Cross-entropy:
https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.cross_entropy.html

Unsqueeze:
https://docs.pytorch.org/docs/2.14/generated/torch.unsqueeze.html

In [ ]:
# Calculate softmax for first row
probabilities = improved_softmax(logits[0])

# Get correct token ID for first row
target_id = y[0]

# Get probability of target token in first row
correct_prob = probabilities[target_id]

# Manual cross-entropy
manual_loss = -torch.log(correct_prob) # negative log of correct token probability

print("Correct target ID:", target_id)
print("Correct-token probability:", correct_prob)
print("\nManual CE loss:", manual_loss)

# PyTorch cross-entropy function
pytorch_loss = F.cross_entropy(
    logits[0].unsqueeze(0), # unsqueeze adds +1 dimensions to tensor, in this case batch dimension
    target_id.unsqueeze(0)
)

print("PyTorch cross-entropy:", pytorch_loss)

Instead of calculating loss for just one token position, we'll do so for all token positions and average their loss scores to get the overall loss of the (psuedo) training example.

Arange:
https://docs.pytorch.org/docs/2.14/generated/torch.arange.html

In [ ]:
# Manual cross-entropy across all token positions (rows)
probabilities = F.softmax(logits, dim=-1)

# Get the probability of the correct target for each row
correct_probs = probabilities[
    torch.arange(len(y)), # tensor of indices from 0 to len(y)
    y # target token IDs
]

# Cross-entropy loss
token_losses = torch.log(correct_probs)

# Average loss across all token positions
manual_sequence_loss = -token_losses.mean()

print("Correct-token probabilities:")
print(correct_probs)

print("\nPer-token losses:")
print(token_losses)

print("\nAverage cross-entropy loss:")
print(manual_sequence_loss)

# PyTorch cross-entropy
pytorch_sequence_loss = F.cross_entropy( # internally applies softmax
    logits, # output logits
    y # target token IDs
)

print("PyTorch loss:", pytorch_sequence_loss)

Perplexity is calculated by exponentiating the cross-entropy loss of a training example. Cross-entropy directly correlates to perplexity, the reason why we exponentiate cross-entropy loss is to convert the log-scaled loss into an intuitive score.

Example:

cross-entropy loss = 0

perplexity = $ e^0 $ = 1

This means the model is effective perfectly confident on predicting the next correct token.

If:

cross-entropy = 2.30

perplexity = 10

The ***intuition*** is that the model behaves loosely as though it's choosing 10 equally likely token options at each step (token position).

In [ ]:
perplexity = torch.exp(pytorch_sequence_loss)

print("Cross-entropy loss:", pytorch_sequence_loss)
print("Perplexity:", perplexity)

We can see this as we artifically raise/lower the logit values of the correct tokens at every position.

In [ ]:
# Copy of logits tensor
better_logits = logits.clone()
worse_logits = logits.clone()

# Add 2.0 to logits for correct tokens in each row
better_logits[
    torch.arange(len(y)),
    y
] += 2.0

# Subtract 2.0 from logits for incorrect tokens in each row
worse_logits[
    torch.arange(len(y)),
    y
] -= 2.0

# Calculate cross-entropy
better_loss = F.cross_entropy(
    better_logits,
    y
)

worse_loss = F.cross_entropy(
    worse_logits,
    y
)

# Exponentiate loss to get perplexity
better_perplexity = torch.exp(better_loss)
worse_perplexity = torch.exp(worse_loss)

print("Original loss:", pytorch_sequence_loss)
print("Original perplexity:", torch.exp(pytorch_sequence_loss))

print("\nBetter loss:", better_loss)
print("Better perplexity:", better_perplexity)

print("\nWorse loss:", worse_loss)
print("Worse perplexity:", worse_perplexity)